# Evaluación histórica del detector inicial

## Objetivo

Analizar el comportamiento histórico del detector experimental:
frecuencia de alertas, cobertura de evaluación y diferencias
entre municipios y periodos.

Se mantiene el método de referencia:
- Comparación con el mismo mes de los cinco años anteriores.
- Mínimo de cinco cambios mensuales históricos.
- Referencia basada en la mediana.
- Escala robusta igual a 1,4826 × MAD.
- Alerta cuando el cambio mensual es positivo y la puntuación
  robusta supera 3.
- Los casos con MAD cero se mantienen como no evaluables.

El análisis comienza en enero de 2011 para comparar años completos
una vez superado el periodo inicial de acumulación de histórico.
El último año puede estar incompleto.

## Alcance y limitaciones

Esta evaluación describe el comportamiento del detector.
Sin una clasificación externa de anomalías confirmadas, no permite
calcular su precisión ni su tasa de falsas alarmas.

Cada referencia utiliza únicamente meses anteriores al evaluado.
Sin embargo, se trabaja con la versión actual del histórico:
no se dispone de las versiones de los datos publicadas en cada fecha.

In [24]:
from pathlib import Path

import pandas as pd
import plotly.express as px

RAIZ = Path.cwd()
if not (RAIZ / "src" / "detectar_alertas.py").is_file():
    RAIZ = RAIZ.parent

RUTA_ALERTAS = RAIZ / "resultados" / "alertas.csv"

if not RUTA_ALERTAS.is_file():
    raise FileNotFoundError(
        "No se encuentra resultados/alertas.csv. "
        "Abre el notebook desde la carpeta del proyecto."
    )

resultados = pd.read_csv(
    RUTA_ALERTAS,
    sep=";",
    encoding="utf-8-sig",
    decimal=".",
    parse_dates=["fecha"],
).sort_values(["municipio", "fecha"]).reset_index(drop=True)

assert not resultados.duplicated(["municipio", "fecha"]).any(), (
    "Hay municipios y meses duplicados."
)
assert resultados["fecha"].notna().all(), "Hay fechas vacías."

estados_validos = {
    "sin_alerta",
    "alerta_experimental",
    "dispersion_cero",
    "historial_insuficiente",
}
assert resultados["estado_detector"].isin(estados_validos).all(), (
    "Hay estados vacíos o desconocidos."
)

print("Filas:", f"{len(resultados):,}".replace(",", "."))
print("Municipios:", resultados["municipio"].nunique())
print(
    "Periodo:",
    resultados["fecha"].min().strftime("%Y-%m"),
    "a",
    resultados["fecha"].max().strftime("%Y-%m"),
)

display(
    resultados[
        ["municipio", "fecha", "paro_total",
         "cambio_mensual", "variacion_mensual_pct"]
    ].head()
)

Filas: 46.898
Municipios: 179
Periodo: 2005-01 a 2026-10


,municipio,fecha,paro_total,cambio_mensual,variacion_mensual_pct
0,"Acebeda, La",2005-01-01,3,NaN,NaN
1,"Acebeda, La",2005-02-01,2,-1.0,-33.333333
2,"Acebeda, La",2005-03-01,2,0.0,0.000000
3,"Acebeda, La",2005-04-01,2,0.0,0.000000
4,"Acebeda, La",2005-05-01,2,0.0,0.000000


In [25]:
INICIO_EVALUACION = pd.Timestamp("2011-01-01")

evaluacion = resultados.loc[
    resultados["fecha"].ge(INICIO_EVALUACION)
].copy()

evaluacion["es_alerta"] = evaluacion["estado_detector"].eq(
    "alerta_experimental"
)
evaluacion["es_evaluable"] = evaluacion["estado_detector"].isin(
    ["sin_alerta", "alerta_experimental"]
)
evaluacion["mad_cero"] = evaluacion["estado_detector"].eq(
    "dispersion_cero"
)
evaluacion["sin_historial"] = evaluacion["estado_detector"].eq(
    "historial_insuficiente"
)

print("Meses analizados:", evaluacion["fecha"].nunique())
display(evaluacion["estado_detector"].value_counts())

Meses analizados: 190


estado_detector
sin_alerta             30162
dispersion_cero         2098
alerta_experimental     1750
Name: count, dtype: int64

In [26]:
resumen_mensual = (
    evaluacion.groupby("fecha", as_index=False)
    .agg(
        municipios=("municipio", "nunique"),
        evaluables=("es_evaluable", "sum"),
        alertas=("es_alerta", "sum"),
        dispersion_cero=("mad_cero", "sum"),
        historial_insuficiente=("sin_historial", "sum"),
    )
)

resumen_mensual["pct_alertas_evaluables"] = (
    100
    * resumen_mensual["alertas"]
    / resumen_mensual["evaluables"].where(
        resumen_mensual["evaluables"].gt(0)
    )
)

resumen_mensual["cobertura_pct"] = (
    100
    * resumen_mensual["evaluables"]
    / resumen_mensual["municipios"]
)

display(
    resumen_mensual.tail(12).style.format(
        {
            "fecha": lambda fecha: fecha.strftime("%Y-%m"),
            "pct_alertas_evaluables": "{:.2f}",
            "cobertura_pct": "{:.2f}",
        },
        decimal=",",
        thousands=".",
        na_rep="No evaluable",
    )
)

,fecha,municipios,evaluables,alertas,dispersion_cero,historial_insuficiente,pct_alertas_evaluables,cobertura_pct
178,2025-11,179,172,9,7,0,"5,23","96,09"
179,2025-12,179,167,6,12,0,"3,59","93,30"
180,2026-01,179,161,4,18,0,"2,48","89,94"
181,2026-02,179,164,9,15,0,"5,49","91,62"
182,2026-03,179,161,11,18,0,"6,83","89,94"
183,2026-04,179,167,4,12,0,"2,40","93,30"
184,2026-05,179,170,7,9,0,"4,12","94,97"
185,2026-06,179,164,13,15,0,"7,93","91,62"
186,2026-07,179,167,19,12,0,"11,38","93,30"
187,2026-08,179,163,20,16,0,"12,27","91,06"


In [27]:
fig_alertas = px.line(
    resumen_mensual,
    x="fecha",
    y="pct_alertas_evaluables",
    title="Municipios con alerta entre los evaluables",
    labels={
        "fecha": "Mes",
        "pct_alertas_evaluables": "Municipios con alerta (%)",
    },
)
fig_alertas.show()

fig_cobertura = px.line(
    resumen_mensual,
    x="fecha",
    y="cobertura_pct",
    title="Cobertura mensual del detector",
    labels={
        "fecha": "Mes",
        "cobertura_pct": "Municipios evaluables (%)",
    },
)
fig_cobertura.update_yaxes(range=[0, 100])
fig_cobertura.show()

## Observaciones sobre el comportamiento histórico

### Frecuencia de alertas

El porcentaje de municipios con alerta entre los evaluables varía
a lo largo del tiempo. Visualmente, en muchos meses queda por debajo
del 10 %, aunque existen episodios con porcentajes superiores.

Destaca un pico alrededor de 2020 que supera el 60 % de los municipios
evaluables. Esto indica que el detector señala aumentos simultáneos
en numerosos municipios. Queda pendiente identificar el mes exacto
y revisar sus cambios observados y referencias históricas.

Este pico no demuestra por sí solo que las alertas sean correctas,
ni permite atribuirles una causa concreta.

En los últimos doce meses mostrados, de noviembre de 2025 a octubre
de 2026, el porcentaje de alertas entre los evaluables oscila entre
el 2,40 % de abril de 2026 y el 12,27 % de agosto de 2026.

En octubre de 2026 se registran 15 alertas entre 172 municipios
evaluables: un 8,72 %. Los otros 157 municipios evaluables no
activan la regla de alerta.

### Cobertura del detector

La cobertura representa el porcentaje de municipios que el método
puede evaluar cada mes. El gráfico muestra una cobertura generalmente
elevada, situada aproximadamente entre el 90 % y el 99 %, con algunas
caídas ligeramente por debajo del 90 %.

En los últimos doce meses mostrados, la cobertura oscila entre
el 89,94 % y el 96,09 %. Hay entre 7 y 18 municipios mensuales
no evaluables por dispersión cero y no aparecen casos con
historial insuficiente en esa tabla.

En octubre de 2026 pueden evaluarse 172 de los 179 municipios
(96,09 %); los 7 restantes presentan MAD cero.

La cobertura mide disponibilidad de evaluación, no precisión
del detector ni porcentaje de población cubierta.

### Tratamiento de los casos con MAD cero

Estos casos se conservan en los resultados con el estado
`dispersion_cero`. No se clasifican como «sin alerta».

No se calcula su puntuación robusta, porque esta requiere dividir
por 1,4826 × MAD. Tampoco se les asigna un límite superior operativo
ni una clasificación de alerta.

Se excluyen del denominador del porcentaje de alertas, pero permanecen
en el total utilizado para calcular la cobertura.

### Limitaciones y comprobaciones pendientes

- La frecuencia de alertas no equivale a una tasa de falsas alarmas.
- Una cobertura elevada no demuestra que las alertas sean correctas.
- La composición de los municipios evaluables puede cambiar cada mes.
- Debemos identificar los meses con mayor concentración de alertas.
- Debemos comprobar qué municipios acumulan casos de MAD cero y
  si el problema se concentra en series con recuentos pequeños.
- Se mantienen los parámetros actuales mientras se analiza
  su comportamiento histórico.

## Revisión del mes con mayor porcentaje de alertas

Identificaremos el mes con mayor porcentaje de municipios con
alerta entre los evaluables y revisaremos:

- El número de alertas y la cobertura de ese mes.
- Los cambios observados frente a las referencias históricas.
- Si las alertas se concentran en determinados municipios.
- El comportamiento en los meses anteriores y posteriores.

El objetivo es comprender por qué se activa el detector.
La coincidencia temporal con un acontecimiento no demuestra
por sí sola una relación causal.

In [28]:
meses_con_mas_alertas = (
    resumen_mensual
    .sort_values(
        ["pct_alertas_evaluables", "fecha"],
        ascending=[False, True],
    )
    .head(10)
)

display(
    meses_con_mas_alertas.style.format(
        {
            "fecha": lambda fecha: fecha.strftime("%Y-%m"),
            "pct_alertas_evaluables": "{:.2f}",
            "cobertura_pct": "{:.2f}",
        },
        decimal=",",
        thousands=".",
        na_rep="No evaluable",
    )
)

,fecha,municipios,evaluables,alertas,dispersion_cero,historial_insuficiente,pct_alertas_evaluables,cobertura_pct
111,2020-04,179,169,114,10,0,"67,46","94,41"
112,2020-05,179,169,58,10,0,"34,32","94,41"
110,2020-03,179,163,42,16,0,"25,77","91,06"
119,2020-12,179,165,31,14,0,"18,79","92,18"
12,2012-01,179,169,31,10,0,"18,34","94,41"
121,2021-02,179,167,28,12,0,"16,77","93,30"
152,2023-09,179,169,25,10,0,"14,79","94,41"
103,2019-08,179,168,23,11,0,"13,69","93,85"
56,2015-09,179,170,23,9,0,"13,53","94,97"
8,2011-09,179,171,21,8,0,"12,28","95,53"


## Interpretación del pico de marzo–mayo de 2020

Los tres meses con mayor porcentaje de municipios con alerta entre
los evaluables son abril, mayo y marzo de 2020.

En marzo se identifican 42 alertas entre 163 municipios evaluables
(25,77 %). En abril se alcanza el máximo, con 114 alertas entre
169 evaluables (67,46 %). En mayo se registran 58 alertas entre
169 evaluables (34,32 %).

El máximo de abril refleja un aumento del número de municipios
que activan la regla, no una reducción del denominador: abril
y mayo cuentan con el mismo número de municipios evaluables.

### Contexto e interpretación

La concentración temporal es coherente con la perturbación laboral
asociada a la COVID-19. La Comunidad de Madrid informó de un aumento
del paro registrado de 41.263 personas en abril de 2020 y relacionó
ese resultado con la paralización de la actividad durante la crisis
del coronavirus.

El detector compara los cambios de cada municipio con los del mismo
mes de los cinco años anteriores. Por tanto, las referencias de
marzo, abril y mayo de 2020 utilizan los meses correspondientes
de 2015–2019, anteriores a la pandemia.

Los resultados aportan evidencia de que el método responde a un
episodio de aumentos generalizados del paro registrado.

### Límites de esta interpretación

- El 67,46 % representa municipios con alerta entre los evaluables:
  no es un porcentaje de aumento del paro ni de personas afectadas.
- Cada municipio tiene el mismo peso en este porcentaje,
  independientemente de su tamaño.
- La coincidencia con un episodio conocido no valida por sí sola
  cada alerta municipal ni el comportamiento global del detector.
- La reducción de alertas en mayo no implica que el paro descendiera:
  indica que menos municipios cumplieron la regla de aumento inusual.
- Los recuentos mensuales no pueden sumarse para obtener municipios
  distintos, porque un municipio puede aparecer en varios meses.

Fuente contextual:
[Comunidad de Madrid, 5 de mayo de 2020](https://www.comunidad.madrid/noticias/2020/05/05/paro-registrado-comunidad-madrid-aumento-41263-personas-mes-abril)

## Revisión municipal de abril de 2020

Revisamos Madrid, Alcalá de Henares y Aranjuez, municipios utilizados
en la exploración previa, sin seleccionarlos por su resultado de alerta.

Para cada municipio:
1. Recuperamos los cambios de marzo a abril de 2015–2019.
2. Recalculamos su mediana y MAD.
3. Reconstruimos el límite y la puntuación de abril de 2020.
4. Comparamos los cálculos con la salida del detector.

Esta comprobación permite explicar y verificar el cálculo.
No constituye una validación de la precisión del detector.

In [29]:
FECHA_REVISION = pd.Timestamp("2020-04-01")
MUNICIPIOS_REVISION = [
    "Madrid",
    "Alcalá de Henares",
    "Aranjuez",
]

historico_revision = resultados.loc[
    resultados["municipio"].isin(MUNICIPIOS_REVISION)
    & resultados["fecha"].dt.month.eq(FECHA_REVISION.month)
    & resultados["fecha"].ge(
        FECHA_REVISION - pd.DateOffset(years=5)
    )
    & resultados["fecha"].lt(FECHA_REVISION),
    ["municipio", "fecha", "cambio_mensual"],
].copy()

historico_revision["anio"] = historico_revision["fecha"].dt.year

tabla_historica = historico_revision.pivot(
    index="municipio",
    columns="anio",
    values="cambio_mensual",
).reindex(MUNICIPIOS_REVISION)

assert tabla_historica.columns.tolist() == list(range(2015, 2020))
assert tabla_historica.notna().all().all(), (
    "Faltan cambios históricos en alguno de los municipios."
)

display(
    tabla_historica.style.format(
        "{:,.0f}",
        decimal=",",
        thousands=".",
    )
)

anio,2015,2016,2017,2018,2019
municipio,,,,,
Madrid,-5.354,-3.863,-5.404,-3.942,-2.837
Alcalá de Henares,-518,-162,-528,-286,-137
Aranjuez,-70,-114,-156,-103,-136


In [30]:
import numpy as np

filas_revision = []

for municipio in MUNICIPIOS_REVISION:
    cambios = tabla_historica.loc[municipio].astype(float)

    referencia = cambios.median()
    mad = (cambios - referencia).abs().median()
    escala = 1.4826 * mad

    fila_actual = resultados.loc[
        resultados["municipio"].eq(municipio)
        & resultados["fecha"].eq(FECHA_REVISION)
    ]

    assert len(fila_actual) == 1, (
        f"No hay una única fila para {municipio} en abril de 2020."
    )
    actual = fila_actual.iloc[0]
    cambio = float(actual["cambio_mensual"])

    if mad > 0:
        limite = referencia + 3 * escala
        puntuacion = (cambio - referencia) / escala
        estado = (
            "alerta_experimental"
            if cambio > 0 and puntuacion > 3
            else "sin_alerta"
        )
    else:
        limite = np.nan
        puntuacion = np.nan
        estado = "dispersion_cero"

    calculos = {
        "referencia_cambio": referencia,
        "mad_cambio": mad,
        "escala_robusta": escala,
        "limite_superior": limite,
        "puntuacion_robusta": puntuacion,
    }

    for columna, calculado in calculos.items():
        guardado = actual[columna]
        coincide = (
            pd.isna(calculado) and pd.isna(guardado)
        ) or (
            pd.notna(calculado)
            and pd.notna(guardado)
            and np.isclose(calculado, float(guardado))
        )
        assert coincide, f"No coincide {columna} en {municipio}."

    assert estado == actual["estado_detector"], (
        f"No coincide el estado en {municipio}."
    )

    filas_revision.append({
        "municipio": municipio,
        "cambio_abril_2020": cambio,
        **calculos,
        "estado_detector": estado,
    })

revision_municipal = pd.DataFrame(filas_revision)

formatos = {
    columna: "{:,.2f}"
    for columna in revision_municipal.select_dtypes(
        include="number"
    ).columns
}

display(
    revision_municipal.style.format(
        formatos,
        decimal=",",
        thousands=".",
        na_rep="No evaluable",
    )
)

print("Los cálculos reconstruidos coinciden con el detector.")

,municipio,cambio_abril_2020,referencia_cambio,mad_cambio,escala_robusta,limite_superior,puntuacion_robusta,estado_detector
0,Madrid,"20.637,00","-3.942,00","1.105,00","1.638,27","972,82","15,00",alerta_experimental
1,Alcalá de Henares,"1.168,00","-286,00","149,00","220,91","376,72","6,58",alerta_experimental
2,Aranjuez,"288,00","-114,00","22,00","32,62","-16,15","12,32",alerta_experimental


Los cálculos reconstruidos coinciden con el detector.


In [31]:
for municipio in MUNICIPIOS_REVISION:
    datos_grafico = resultados.loc[
        resultados["municipio"].eq(municipio)
        & resultados["fecha"].dt.month.eq(4)
        & resultados["fecha"].ge(pd.Timestamp("2015-04-01"))
        & resultados["fecha"].le(FECHA_REVISION),
        ["fecha", "cambio_mensual"],
    ].copy()

    datos_grafico["anio"] = (
        datos_grafico["fecha"].dt.year.astype(str)
    )
    datos_grafico["tipo"] = np.where(
        datos_grafico["fecha"].eq(FECHA_REVISION),
        "Mes evaluado",
        "Histórico de referencia",
    )

    revision = revision_municipal.loc[
        revision_municipal["municipio"].eq(municipio)
    ].iloc[0]

    fig = px.bar(
        datos_grafico,
        x="anio",
        y="cambio_mensual",
        color="tipo",
        color_discrete_map={
            "Histórico de referencia": "#4477AA",
            "Mes evaluado": "#EE7733",
        },
        title=f"{municipio}: cambios de marzo a abril",
        labels={
            "anio": "Año",
            "cambio_mensual": "Cambio del paro registrado (personas)",
            "tipo": "",
        },
    )

    fig.add_hline(
        y=0,
        line_color="gray",
        line_width=1,
    )
    fig.add_hline(
        y=revision["referencia_cambio"],
        line_dash="dot",
        annotation_text="Mediana 2015–2019",
    )

    if pd.notna(revision["limite_superior"]):
        fig.add_hline(
            y=revision["limite_superior"],
            line_dash="dash",
            line_color="red",
            annotation_text="Límite para abril de 2020",
        )

    fig.show()

## Conclusiones de la revisión municipal de abril de 2020

En Madrid, Alcalá de Henares y Aranjuez, los cambios de marzo a abril
fueron negativos en los cinco años de referencia, de 2015 a 2019.
En abril de 2020, los tres municipios registraron aumentos positivos
que superaron sus respectivos límites.

- Madrid: aumento de 20.637 personas frente a una mediana histórica
  de −3.942 y un límite de 972,82. Puntuación robusta: 15,00.
- Alcalá de Henares: aumento de 1.168 personas frente a una mediana
  de −286 y un límite de 376,72. Puntuación robusta: 6,58.
- Aranjuez: aumento de 288 personas frente a una mediana de −114
  y un límite de −16,15. Puntuación robusta: 12,32.

Los tres casos cumplen las dos condiciones de alerta:
cambio mensual positivo y puntuación robusta superior a 3.

### Interpretación

El detector identifica un cambio de comportamiento respecto a los
cinco abriles anteriores: de descensos del paro a aumentos superiores
al límite de referencia.

Las puntuaciones no constituyen una clasificación de gravedad social.
Aranjuez presenta una puntuación superior a Alcalá de Henares pese
a registrar un aumento menor, porque su dispersión histórica también
es menor.

El límite negativo de Aranjuez refleja una referencia de descensos
habituales. Se mantiene la condición adicional de cambio positivo
para no alertar ante simples descensos menores de lo habitual.

### Alcance

Estos resultados permiten explicar el funcionamiento del detector
en tres casos concretos del episodio de 2020. No demuestran su
precisión general ni permiten estimar falsas alarmas.

Queda pendiente revisar la cobertura por municipio, especialmente
la recurrencia de MAD cero, y estudiar casos fuera de este episodio.

## Recurrencia de MAD cero por municipio

Calculamos, desde enero de 2011, cuántos meses presenta cada municipio
con MAD cero y qué porcentaje representan sobre sus meses observados.

También mostramos los meses evaluables y la mediana del paro registrado
durante el periodo, como descripción del volumen de la serie.

El objetivo es identificar limitaciones persistentes de cobertura.
No se modifican los parámetros ni se asignan alertas a los casos
no evaluables.

In [32]:
resumen_municipal = (
    evaluacion.groupby("municipio", as_index=False)
    .agg(
        meses_observados=("fecha", "nunique"),
        meses_evaluables=("es_evaluable", "sum"),
        meses_mad_cero=("mad_cero", "sum"),
        meses_sin_historial=("sin_historial", "sum"),
        meses_con_alerta=("es_alerta", "sum"),
        mediana_paro_total=("paro_total", "median"),
    )
)

resumen_municipal["pct_mad_cero"] = (
    100
    * resumen_municipal["meses_mad_cero"]
    / resumen_municipal["meses_observados"]
)

resumen_municipal["cobertura_pct"] = (
    100
    * resumen_municipal["meses_evaluables"]
    / resumen_municipal["meses_observados"]
)

resumen_municipal["pct_alertas_evaluables"] = (
    100
    * resumen_municipal["meses_con_alerta"]
    / resumen_municipal["meses_evaluables"].where(
        resumen_municipal["meses_evaluables"].gt(0)
    )
)

ranking_mad_cero = resumen_municipal.sort_values(
    ["pct_mad_cero", "municipio"],
    ascending=[False, True],
)

columnas_cobertura = [
    "municipio",
    "meses_observados",
    "meses_mad_cero",
    "pct_mad_cero",
    "meses_sin_historial",
    "cobertura_pct",
    "mediana_paro_total",
]

display(
    ranking_mad_cero[columnas_cobertura].head(20).style.format(
        {
            "pct_mad_cero": "{:.2f}",
            "cobertura_pct": "{:.2f}",
            "mediana_paro_total": "{:,.1f}",
        },
        decimal=",",
        thousands=".",
        na_rep="No evaluable",
    )
)


,municipio,meses_observados,meses_mad_cero,pct_mad_cero,meses_sin_historial,cobertura_pct,mediana_paro_total
67,"Hiruela, La",190,163,"85,79",0,"14,21","1,0"
121,Robregordo,190,125,"65,79",0,"34,21","3,0"
76,Madarcos,190,122,"64,21",0,"35,79","2,0"
69,Horcajuelo de la Sierra,190,119,"62,63",0,"37,37","3,0"
0,"Acebeda, La",190,96,"50,53",0,"49,47","5,0"
112,Puebla de la Sierra,190,91,"47,89",0,"52,11","5,0"
136,Somosierra,190,87,"45,79",0,"54,21","4,0"
14,"Atazar, El",190,83,"43,68",0,"56,32","9,0"
111,Prádena del Rincón,190,71,"37,37",0,"62,63","5,0"
119,Robledillo de la Jara,190,62,"32,63",0,"67,37","7,0"


In [33]:
datos_mad_grafico = (
    ranking_mad_cero.head(15)
    .sort_values("pct_mad_cero")
)

fig = px.bar(
    datos_mad_grafico,
    x="pct_mad_cero",
    y="municipio",
    orientation="h",
    title="Municipios con mayor porcentaje de meses con MAD cero",
    labels={
        "pct_mad_cero": "Meses con MAD cero (%)",
        "municipio": "Municipio",
    },
    hover_data=[
        "meses_mad_cero",
        "meses_observados",
        "mediana_paro_total",
    ],
)

fig.update_xaxes(range=[0, 100])
fig.update_layout(height=650)
fig.show()

In [34]:
import plotly.express as px

datos_dispersion = resumen_municipal.copy()

# La escala logarítmica necesita valores positivos.
assert datos_dispersion["mediana_paro_total"].notna().all(), (
    "Hay medianas de paro vacías."
)
assert datos_dispersion["mediana_paro_total"].gt(0).all(), (
    "Hay medianas de paro iguales a cero o negativas. "
    "Hay que revisar esos casos antes de usar la escala logarítmica."
)

def formato_es(valor, decimales=2):
    return (
        f"{valor:,.{decimales}f}"
        .replace(",", "_")
        .replace(".", ",")
        .replace("_", ".")
    )

# Texto de presentación: no modifica los valores numéricos.
datos_dispersion["mediana_texto"] = datos_dispersion[
    "mediana_paro_total"
].map(lambda valor: formato_es(valor, 1))

datos_dispersion["mad_cero_texto"] = datos_dispersion[
    "pct_mad_cero"
].map(lambda valor: formato_es(valor, 2))

fig = px.scatter(
    datos_dispersion,
    x="mediana_paro_total",
    y="pct_mad_cero",
    log_x=True,
    custom_data=[
        "municipio",
        "mediana_texto",
        "mad_cero_texto",
        "meses_mad_cero",
        "meses_observados",
    ],
    labels={
        "mediana_paro_total": (
            "Mediana del paro registrado "
            "(personas; escala logarítmica)"
        ),
        "pct_mad_cero": "Meses con MAD cero (%)",
    },
    title="Volumen de paro registrado y frecuencia de MAD cero",
    template="plotly_white",
)

fig.update_traces(
    marker={
        "size": 9,
        "opacity": 0.7,
        "color": "#4477AA",
    },
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Mediana del paro: %{customdata[1]} personas<br>"
        "Meses con MAD cero: %{customdata[3]}"
        " de %{customdata[4]}<br>"
        "Porcentaje: %{customdata[2]} %"
        "<extra></extra>"
    ),
)

fig.update_xaxes(
    tickmode="array",
    tickvals=[1, 10, 100, 1_000, 10_000, 100_000],
    ticktext=["1", "10", "100", "1.000", "10.000", "100.000"],
    showgrid=True,
    title_standoff=18,
)

fig.update_yaxes(
    range=[-2, 100],
    tickmode="array",
    tickvals=[0, 20, 40, 60, 80, 100],
    ticksuffix=" %",
    title_standoff=15,
)

fig.update_layout(
    height=550,
    margin={"l": 80, "r": 30, "t": 80, "b": 90},
    font={"size": 13},
    hoverlabel={"font_size": 13},
)

fig.show()

## Conclusiones sobre los casos con MAD cero

La falta de evaluación por MAD cero no se distribuye uniformemente
entre los municipios. Se concentra especialmente en las series
con volúmenes pequeños de paro registrado.

Durante los 190 meses analizados:

- La Hiruela presenta MAD cero en 163 meses (85,79 %), con una
  cobertura del 14,21 % y una mediana de paro registrado de 1 persona.
- Robregordo presenta MAD cero en 125 meses (65,79 %), con una
  mediana de 3 personas.
- Madarcos presenta MAD cero en 122 meses (64,21 %), con una
  mediana de 2 personas.
- Horcajuelo de la Sierra presenta MAD cero en 119 meses (62,63 %).
- La Acebeda presenta MAD cero en 96 meses (50,53 %).

En estos casos, la limitación no procede de la falta de histórico:
la tabla muestra cero meses con historial insuficiente.

### Relación con el volumen de la serie

El gráfico de dispersión muestra una asociación entre un menor
volumen de paro registrado y una mayor frecuencia de MAD cero.
En las series de mayor volumen, la frecuencia observada es
generalmente baja o nula.

Este patrón es coherente con la naturaleza discreta de los recuentos:
cuando hay pocas personas, los cambios mensuales pueden repetir
frecuentemente los mismos valores.

Con cinco observaciones históricas, basta con que al menos tres
cambios coincidan para que la MAD sea cero. Esto no implica que
todos los cambios sean iguales ni que el paro no varíe.

La relación observada es descriptiva: no establece un umbral
de tamaño a partir del cual el detector funcione correctamente.

### Implicaciones para el proyecto

La cobertura agregada elevada no garantiza una cobertura adecuada
para cada municipio. El método actual presenta una limitación
persistente en algunas series con recuentos pequeños.

Los casos con MAD cero deben mantenerse visibles como no evaluables,
conservando el paro registrado y su cambio mensual. No deben
presentarse como municipios sin alerta.

La mediana del paro utilizada en este gráfico resume todo el periodo
y tiene una función descriptiva. No se incorpora al cálculo histórico
de las alertas.

Estos resultados justifican evaluar alternativas, pero todavía
no permiten elegir una solución ni sustituir la MAD por una
constante arbitraria.